In [1]:
import torch 
from torchvision.models import resnet18, ResNet18_Weights

model = resnet18(weights = ResNet18_Weights)

/home/nishant_linux_pro/Desktop/Remote_Sensing/.venv/lib/python3.12/site-packages/torchvision/models/_utils.py:222: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [2]:
import torch.nn as nn 
old_conv = model.conv1

new_conv = nn.Conv2d(
    in_channels=13,
    out_channels=64,
    kernel_size=7,
    stride=2,
    padding=3,
    bias=False
)

with torch.no_grad(): 

    new_conv.weight[:, 1] = old_conv.weight[:, 2]  # B2 = Blue
    new_conv.weight[:, 2] = old_conv.weight[:, 1]  # B3 = Green
    new_conv.weight[:, 3] = old_conv.weight[:, 0]  # B4 = Red

    mean_weight = old_conv.weight.mean(dim=1, keepdim=True)

    for i in [0, 4, 5, 6, 7, 8, 9, 10, 11, 12]:
        new_conv.weight[:, i] = (
            mean_weight[:, 0] +
            0.01 * torch.randn_like(mean_weight[:, 0])
        )
model.fc = nn.Linear(in_features=model.fc.in_features,out_features=10)

model.conv1 = new_conv

In [3]:
for param in model.parameters():
    param.requires_grad = False

for param in model.conv1.parameters():
    param.requires_grad = True

for param in model.fc.parameters():
    param.requires_grad = True

for name, param in model.named_parameters():
    if param.requires_grad:
        print(name, "       : unfreezed")

    

conv1.weight        : unfreezed
fc.weight        : unfreezed
fc.bias        : unfreezed


In [4]:
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-3,
    weight_decay=1e-4
)


In [ ]:
import torch 
import torch.nn as nn 
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import matplotlib.pyplot as plt

if torch.cuda.is_available():
    device = 'cuda'
else:
    device = 'cpu'

def load_TTCV():

    X_test = torch.load("X_test.pt", weights_only=False)
    Y_test = torch.load("Y_test.pt", weights_only=False)

    X_test = torch.stack([
        torch.tensor(img, dtype=torch.float32) for img in X_test
    ])

    X_cv = torch.load("X_cv.pt", weights_only=False)
    Y_cv = torch.load("Y_cv.pt", weights_only=False)

    X_cv = torch.stack([
        torch.tensor(img, dtype=torch.float32) for img in X_cv
    ])

    X_train = ds['image']
    Y_train = ds['filename']

    Y_train = [name.split('/')[0] for name in Y_train]

    X_train = torch.stack([
        torch.tensor(img, dtype=torch.float32) for img in X_train
    ])

    classes = [
        'AnnualCrop',
        'Forest',
        'HerbaceousVegetation',
        'Highway',
        'Industrial',
        'Pasture',
        'PermanentCrop',
        'Residential',
        'River',
        'SeaLake'
    ]

    class_to_idx = {
        name: idx
        for idx, name in enumerate(classes)
    }

    idx_to_class = {
        idx: name
        for name, idx in class_to_idx.items()
    }

    Y_train = torch.tensor(
        [class_to_idx[name] for name in Y_train],
        dtype=torch.long
    )

    train_dataset = TensorDataset(X_train, Y_train)
    train_loader = DataLoader(
        train_dataset,
        batch_size=64,
        shuffle=True
    )

    cv_dataset = TensorDataset(X_cv, Y_cv)
    cv_loader = DataLoader(
        cv_dataset,
        batch_size=64,
        shuffle=False
    )

    test_dataset = TensorDataset(X_test, Y_test)
    test_loader = DataLoader(
        test_dataset,
        batch_size=64,
        shuffle=False
    )

    Mean = [
        0.13543268, 0.11175188, 0.10418837, 0.09467538,
        0.11996479, 0.20028183, 0.23731175, 0.23004931,
        0.07332496, 0.0012104, 0.18213873, 0.11190669,
        0.25987505
    ]

    Std = [
        0.02457886, 0.03333259, 0.0395221, 0.05942802,
        0.05672322, 0.08590868, 0.10834781, 0.11147044,
        0.04039885, 0.00047205, 0.10015353, 0.07607304,
        0.12276264
    ]

    Mean = torch.tensor(Mean, dtype=torch.float32)
    Std = torch.tensor(Std, dtype=torch.float32)

    return (X_train, Y_train,X_cv, Y_cv,X_test, Y_test,train_loader,cv_loader,test_loader,class_to_idx,idx_to_class,Mean,Std)
 


Mean = [
    0.13543268, 0.11175188, 0.10418837, 0.09467538, 0.11996479,
    0.20028183, 0.23731175, 0.23004931, 0.07332496, 0.0012104,
    0.18213873, 0.11190669, 0.25987505
]

Std = [
    0.02457886, 0.03333259, 0.0395221, 0.05942802, 0.05672322,
    0.08590868, 0.10834781, 0.11147044, 0.04039885, 0.00047205,
    0.10015353, 0.07607304, 0.12276264
]

Mean = torch.tensor(Mean, dtype=torch.float32)
Std = torch.tensor(Std, dtype=torch.float32)

classes = [
    'AnnualCrop',
    'Forest',
    'HerbaceousVegetation',
    'Highway',
    'Industrial',
    'Pasture',
    'PermanentCrop',
    'Residential',
    'River',
    'SeaLake'
]




KeyboardInterrupt: 

In [ ]:
from torch.utils.data import DataLoader, TensorDataset
from datasets import load_from_disk

                                    # ds = load_from_disk("EuroSAT_MS_train")
                                    # X_train = ds['image']
                                    # Y_train = ds['filename']
                                    # Y_train = [name.split('/')[0] for name in Y_train]
                                    # X_train = torch.stack([
                                    #     torch.tensor(img, dtype=torch.float32)
                                    #     for img in X_train
                                    # ])
                                    # unique_ids = sorted(set(Y_train))

                                    # class_to_idx = {
                                    #     name: idx
                                    #     for idx, name in enumerate(unique_ids)
                                    # }

                                    # Y_train = torch.tensor(
                                    #     [class_to_idx[name] for name in Y_train],
                                    #     dtype=torch.long
                                    # )

                                    # tensor_ds = TensorDataset(X_train, Y_train)

                                    # tensor_ds_loader = DataLoader(tensor_ds, batch_size = 64, shuffle = True)


Mean = [0.13543268, 0.11175188, 0.10418837, 0.09467538, 0.11996479, 0.20028183,
        0.23731175, 0.23004931, 0.07332496, 0.0012104, 0.18213873, 0.11190669,
        0.25987505]

Std = [0.02457886, 0.03333259, 0.0395221, 0.05942802, 0.05672322, 0.08590868,
       0.10834781, 0.11147044, 0.04039885, 0.00047205, 0.10015353, 0.07607304,
       0.12276264]

Mean = torch.tensor(Mean, dtype=torch.float32)
Std = torch.tensor(Std, dtype=torch.float32)

idx_to_class = {v: k for k, v in class_to_idx.items()}


In [ ]:
num_epochs = 5

model = model.to('cuda')

loss_fn = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(),lr=0.001)

for epoch in range(num_epochs):
 
    model.train()

    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for X, Y in tensor_ds_loader:

        X = X.to(device)
        Y = Y.to(device)

        optimizer.zero_grad()

        output = model(X)

        loss = loss_fn(output, Y)

        loss.backward()

        optimizer.step()

        train_loss += loss.item() * X.size(0)

        predicted = output.argmax(dim=1)

        train_correct += (predicted == Y).sum().item()
        train_total += Y.size(0)

    train_loss /= train_total
    train_accuracy = train_correct / train_total


    model.eval()

    cv_loss = 0.0
    cv_correct = 0
    cv_total = 0

    with torch.no_grad():

        for X, Y in cv_loader:

            X = X.to(device)
            Y = Y.to(device)

            output = model(X)

            loss = loss_fn(output, Y)

            cv_loss += loss.item() * X.size(0)

            predicted = output.argmax(dim=1)

            cv_correct += (predicted == Y).sum().item()
            cv_total += Y.size(0)

    cv_loss /= cv_total
    cv_accuracy = cv_correct / cv_total


    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"Train Loss: {train_loss:.4f} "
        f"Train Acc: {train_accuracy:.4f} "
        f"CV Loss: {cv_loss:.4f} "
        f"CV Acc: {cv_accuracy:.4f}"
    )

Epoch [1/5] Train Loss: 0.8367 Train Acc: 0.7437 CV Loss: 0.4668 CV Acc: 0.8561
Epoch [2/5] Train Loss: 0.4512 Train Acc: 0.8581 CV Loss: 0.4110 CV Acc: 0.8686
Epoch [3/5] Train Loss: 0.3868 Train Acc: 0.8782 CV Loss: 0.3522 CV Acc: 0.8929
Epoch [4/5] Train Loss: 0.3541 Train Acc: 0.8839 CV Loss: 0.3430 CV Acc: 0.8953
Epoch [5/5] Train Loss: 0.3354 Train Acc: 0.8909 CV Loss: 0.3375 CV Acc: 0.8953


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

cv_loss = [0.4865, 0.3914, 0.3866, 0.3725, 0.3627]
train_acc = [0.7514, 0.8574, 0.8734, 0.8828, 0.8898]
cv_acc = [0.8490, 0.8786, 0.8784, 0.8805, 0.8797]
train_loss = [0.8079, 0.4435, 0.3895, 0.3610, 0.3419]

e = np.arange(1, 6)

# Plot loss
plt.plot(e, cv_loss, label="CV Loss", c='g')
plt.plot(e, train_loss, label="Train Loss", c='r')

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs CV Loss")
plt.legend()
plt.grid(True)
plt.show()
plt.plot(e, cv_acc, label="CV Accuracy", c='g')
plt.plot(e, train_acc, label="Train Accuracy", c='r')

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training vs CV Accuracy")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
for param in model.parameters():
    param.requires_grad = True

optimizer = torch.optim.AdamW([
    {"params": model.conv1.parameters(), "lr": 1e-4},
    {"params": model.layer1.parameters(), "lr": 1e-4},
    {"params": model.layer2.parameters(), "lr": 1e-4},
    {"params": model.layer3.parameters(), "lr": 1e-4},
    {"params": model.layer4.parameters(), "lr": 1e-4},
    {"params": model.fc.parameters(), "lr": 1e-3}
], weight_decay=1e-4)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=15
)

In [ ]:
num_epochs = 15

train_losses = []
cv_losses = []
train_accs = []
cv_accs = []

best_cv_accuracy = 0.0
X_train, Y_train,X_cv, Y_cv,X_test, Y_test,train_loader,cv_loader,test_loader,class_to_idx,idx_to_class,Mean,Std = load_TTCV()
for epoch in range(num_epochs):
 

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for X, Y in train_loader:

        X = X.to(device)
        Y = Y.to(device)

        optimizer.zero_grad()

        output = model(X)

        loss = loss_fn(output, Y)

        loss.backward()

        optimizer.step()

        running_loss += loss.item() * X.size(0)

        predicted = output.argmax(dim=1)

        correct += (predicted == Y).sum().item()
        total += Y.size(0)

    train_loss = running_loss / total
    train_accuracy = correct / total
 
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():

        for X, Y in cv_loader:

            X = X.to(device)
            Y = Y.to(device)

            output = model(X)

            loss = loss_fn(output, Y)

            running_loss += loss.item() * X.size(0)

            predicted = output.argmax(dim=1)

            correct += (predicted == Y).sum().item()
            total += Y.size(0)

    cv_loss = running_loss / total
    cv_accuracy = correct / total

 
    train_losses.append(train_loss)
    cv_losses.append(cv_loss)

    train_accs.append(train_accuracy)
    cv_accs.append(cv_accuracy)

 
    if cv_accuracy > best_cv_accuracy:

        best_cv_accuracy = cv_accuracy

        torch.save(
            model.state_dict(),
            "resnet18_finetuned_best.pt"
        )

 
    scheduler.step()


    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"Train Loss: {train_loss:.4f} "
        f"Train Acc: {train_accuracy:.4f} "
        f"CV Loss: {cv_loss:.4f} "
        f"CV Acc: {cv_accuracy:.4f}"
    )

Epoch [1/15] Train Loss: 0.2232 Train Acc: 0.9300 CV Loss: 0.1197 CV Acc: 0.9602
Epoch [2/15] Train Loss: 0.0932 Train Acc: 0.9694 CV Loss: 0.1448 CV Acc: 0.9604
Epoch [3/15] Train Loss: 0.0527 Train Acc: 0.9832 CV Loss: 0.1407 CV Acc: 0.9621
Epoch [4/15] Train Loss: 0.0368 Train Acc: 0.9876 CV Loss: 0.1206 CV Acc: 0.9687
Epoch [5/15] Train Loss: 0.0265 Train Acc: 0.9912 CV Loss: 0.1165 CV Acc: 0.9704
Epoch [6/15] Train Loss: 0.0212 Train Acc: 0.9930 CV Loss: 0.1045 CV Acc: 0.9715
Epoch [7/15] Train Loss: 0.0129 Train Acc: 0.9957 CV Loss: 0.1041 CV Acc: 0.9748
Epoch [8/15] Train Loss: 0.0108 Train Acc: 0.9966 CV Loss: 0.1077 CV Acc: 0.9759
Epoch [9/15] Train Loss: 0.0066 Train Acc: 0.9978 CV Loss: 0.1071 CV Acc: 0.9734
Epoch [10/15] Train Loss: 0.0061 Train Acc: 0.9983 CV Loss: 0.0986 CV Acc: 0.9750
Epoch [11/15] Train Loss: 0.0030 Train Acc: 0.9992 CV Loss: 0.0939 CV Acc: 0.9743
Epoch [12/15] Train Loss: 0.0017 Train Acc: 0.9995 CV Loss: 0.0923 CV Acc: 0.9771
Epoch [13/15] Train Loss:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

cv_loss = [0.1429, 0.1148, 0.1396, 0.1422, 0.1394, 0.1363, 0.1310, 0.1139, 0.1093, 0.1163, 0.1127, 0.1104, 0.1106, 0.1073, 0.1086]
train_acc = [0.9288, 0.9716, 0.9820, 0.9892, 0.9897, 0.9915, 0.9951, 0.9969, 0.9979, 0.9988, 0.9996, 0.9993, 0.9992, 0.9998, 0.9998]
cv_acc = [0.9558, 0.9648, 0.9623, 0.9632, 0.9660, 0.9671, 0.9673, 0.9737, 0.9741, 0.9721, 0.9732, 0.9750, 0.9737, 0.9739, 0.9754]
train_loss = [0.2326, 0.0900, 0.0547, 0.0325, 0.0328, 0.0256, 0.0174, 0.0105, 0.0063, 0.0035, 0.0018, 0.0023, 0.0029, 0.0009, 0.0010]

e = np.arange(1, 16)

plt.plot(e, cv_loss, label="CV Loss", c='g')
plt.plot(e, train_loss, label="Train Loss", c='r')

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs CV Loss")
plt.legend()
plt.grid(True)
plt.show()
plt.plot(e, cv_acc, label="CV Accuracy", c='g')
plt.plot(e, train_acc, label="Train Accuracy", c='r')

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training vs CV Accuracy")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
X_train, Y_train,X_cv, Y_cv,X_test, Y_test,train_loader,cv_loader,test_loader,class_to_idx,idx_to_class,Mean,Std

In [ ]:
# Load architecture + checkpoint
model_dict = torch.load("resnet18_finetuned_best.pt", weights_only=True)

model = resnet18()

model.conv1 = nn.Conv2d(13, 64, 7, 2, 3, bias=False)
model.fc = nn.Linear(512, 10)

model.load_state_dict(model_dict)
model = model.to(device)
model.eval()

def predict(i):
    img = X_test[i]

    img_view = img * Std[:, None, None] + Mean[:, None, None]

    rgb = img_view[[3, 2, 1]]
    rgb = rgb.permute(1, 2, 0)

    rgb = torch.clamp(rgb, 0, 1)

    plt.imshow(rgb)
    plt.axis("off")
    plt.show()

    img = img.unsqueeze(0).to(device)

    model.eval()

    with torch.no_grad():

        output = model(img)
        prob = torch.softmax(output, dim=1)

    predicted_idx = prob.argmax(dim=1).item()
    actual_idx = Y_test[i].item()

    print("Actual:", classes[actual_idx])
    print("Predicted:", classes[predicted_idx])
    print("Probability:", prob[0, predicted_idx].item())


for i in range(1000):

    predict(i*150)

